In [ ]:
#conversacion con gemini
# https://gemini.google.com/app/2622fa6806eb26a5

import os
import subprocess
try:
    _printenv = subprocess.run(
        ['bash', '-c', 'source ~/.bashrc 2>/dev/null && printenv'],
        text=True, capture_output=True, timeout=10,
    ).stdout
    for _line in _printenv.splitlines():
        if '=' in _line:
            _k, _v = _line.split('=', 1)
            os.environ.setdefault(_k, _v)
except Exception:
    pass
if 'PDK_ROOT' in os.environ and 'PDK' in os.environ:
    os.environ.setdefault('PDKPATH', os.path.join(os.environ['PDK_ROOT'], os.environ['PDK']))

In [ ]:
#Variables de entorno dentro del contenedor del docker
print(_printenv)

In [ ]:
#funciones para ir desplegando el gds y desplegar los componentes en jupiter

import gdstk
import svgutils.transform as sg
import IPython.display
from IPython.display import clear_output
import ipywidgets as widgets

# Redirect all outputs here
hide = widgets.Output()

def display_gds(gds_file,path,scale = 3):
  
  # Generate an SVG image
  top_level_cell = gdstk.read_gds(gds_file).top_level()[0]
  top_level_cell.write_svg(os.path.join(path,'out.svg'))
    
  # Scale the image for displaying
  fig = sg.fromfile(os.path.join(path,'out.svg'))
  fig.set_size((str(float(fig.width) * scale), str(float(fig.height) * scale)))
  fig.save(os.path.join(path,'out.svg'))

  # Display the image
  IPython.display.display(IPython.display.SVG(os.path.join(path,'out.svg')))
  os.remove(os.path.join(path,'out.gds'))

def display_component(component,path,scale = 3):
  # Save to a GDS file
  with hide:
    component.write_gds(os.path.join(path,'out.gds'))
  display_gds(os.path.join(path,'out.gds'),path,scale)

In [ ]:
# %%

import os
import gdsfactory as gf #duda
from gdsfactory import Component
from glayout import MappedPDK, gf180
from gdsfactory.components import text_freetype, rectangle
from glayout import nmos, pmos
from glayout.primitives.mimcap import mimcap, mimcap_array #Genera un capacitor tipo MIM (Metal-Insulator-Metal)
from glayout.routing.straight_route import straight_route
from glayout.routing.c_route import c_route
from glayout.routing.L_route import L_route
from glayout.util.comp_utils import align_comp_to_port, evaluate_bbox, prec_center, prec_ref_center
from glayout.util.port_utils import add_ports_perimeter, print_ports
from glayout.util.snap_to_grid import component_snap_to_grid
from glayout.spice.netlist import Netlist
from glayout import via_stack
from glayout import rename_ports_by_orientation
from glayout import tapring

In [ ]:


integrator_config = {
    "pdk": gf180,
    "layout_rules": {
        "spacing": gf180.util_max_metal_seperation(),
        "routing_metal": "met2",
        "dummy_devices": True,
        "tie_layers": ("met2", "met1"),
        "sd_rmult": 1,
    },
}

# Base parameters for PMOS and NMOS
nmos_kwargs = {
    "with_tie": True, #checar a true
    "with_dnwell": False,
    "sd_route_topmet": "met2",
    "gate_route_topmet": "met2",
    "sd_route_left": True,
    "rmult": None,
    "gate_rmult": 1,
    "interfinger_rmult": 1,
    "substrate_tap_layers": ("met2", "met1"),
    "dummy_routes": False,
}

pmos_kwargs = {
    "with_tie": True,#chacar a True
    "dnwell": False,
    "sd_route_topmet": "met2",
    "gate_route_topmet": "met2",
    "sd_route_left": True,
    "rmult": None,
    "gate_rmult": 1,
    "interfinger_rmult": 1,
    "substrate_tap_layers": ("met2", "met1"),
    "dummy_routes": False,
}

In [ ]:
# .subckt integrator_final_mike Vext_pin avdd avss I_50n
# *.PININFO Vext_pin:I avdd:B avss:B I_50n:I
# XM1 vg vg vm avdd pfet_03v3 L=.28u W=1u nf=1 ad='int((nf+1)/2) * W/nf * 0.18u' as='int((nf+2)/2) * W/nf * 0.18u' pd='2*int((nf+1)/2) * (W/nf + 0.18u)'
# + ps='2*int((nf+2)/2) * (W/nf + 0.18u)' nrd='0.18u / W' nrs='0.18u / W' sa=0 sb=0 sd=0
# XM6 vm Vext_pin avdd avss nfet_03v3 L=0.28u W=1u nf=1 ad='int((nf+1)/2) * W/nf * 0.18u' as='int((nf+2)/2) * W/nf * 0.18u' pd='2*int((nf+1)/2) * (W/nf + 0.18u)'
# + ps='2*int((nf+2)/2) * (W/nf + 0.18u)' nrd='0.18u / W' nrs='0.18u / W' sa=0 sb=0 sd=0
# XM2 avss I_50n vg avss nfet_03v3 L=0.28u W=1u nf=1 ad='int((nf+1)/2) * W/nf * 0.18u' as='int((nf+2)/2) * W/nf * 0.18u' pd='2*int((nf+1)/2) * (W/nf + 0.18u)'
# + ps='2*int((nf+2)/2) * (W/nf + 0.18u)' nrd='0.18u / W' nrs='0.18u / W' sa=0 sb=0 sd=0
# XM3 I_50n I_50n avss avss nfet_03v3 L=0.28u W=1u nf=1 ad='int((nf+1)/2) * W/nf * 0.18u' as='int((nf+2)/2) * W/nf * 0.18u' pd='2*int((nf+1)/2) * (W/nf + 0.18u)'
# + ps='2*int((nf+2)/2) * (W/nf + 0.18u)' nrd='0.18u / W' nrs='0.18u / W' sa=0 sb=0 sd=0
# XC3 vm avss cap_mim_2f0fF c_width=17.68e-6 c_length=17.68e-6 m=8
# .ends


In [ ]:
# ----------------------------------------------------------------------
# 2. Instantiate Devices
# ----------------------------------------------------------------------
pdk = integrator_config["pdk"]
integrator = Component(name="integrator")

# PMOS Transistors (XM1, XM6)
xm1 = pmos(pdk, width=1.0, length=0.28, fingers=1,  with_dummy=(False, False), with_substrate_tap=False, **pmos_kwargs)


# NMOS Transistors (XM2, XM3)
xm2 = nmos(pdk, width=1.0, length=0.28, fingers=1, with_dummy=(False, False), with_substrate_tap=False,  **nmos_kwargs)
xm3 = nmos(pdk, width=1.0, length=0.28, fingers=1,  with_dummy=(False, False), with_substrate_tap=False, **nmos_kwargs)
xm6 = nmos(pdk, width=1.0, length=0.28, fingers=1,  with_dummy=(False, False), with_substrate_tap=False, **nmos_kwargs)


# MIM Capacitor Array XC3 (m=8, 2 rows x 4 columns)
xc3 = mimcap_array(
    pdk, 
    size=(17.68, 17.68), 
    rows=2, 
    columns=4
)

#nombrandalos para que aparezcan en el layout

xm1.name ="xm1"
xm6.name ="xm6"
xm2.name ="xm2"
xm3.name ="xm3"
xc3.name ="xc3"

# Add components as references to the main cell
xm1_ref = integrator << xm1
xm6_ref = integrator << xm6
xm2_ref = integrator << xm2
xm3_ref = integrator << xm3
xc3_ref = integrator << xc3



xm1_xy = evaluate_bbox(xm1)
xm2_xy = evaluate_bbox(xm2)
xm3_xy = evaluate_bbox(xm3)
xm6_xy = evaluate_bbox(xm6)
xc3_xy = evaluate_bbox(xc3)


# ----------------------------------------------------------------------
# 3. Component Placement & Floorplanning
# ----------------------------------------------------------------------
# Row 1 (Top): PMOS transistors side-by-side
# xm1_ref.move((15, 24))
sepa=5
xm6_ref.movex(xm6_xy[0]/2)
xm3_ref.movex(xm6_ref.xmax + xm3_xy[0]/2 + pdk.util_max_metal_seperation() + sepa)
xm2_ref.movex(xm3_ref.xmax + xm2_xy[0]/2 + pdk.util_max_metal_seperation() + sepa)
xc3_ref.movex(xm2_ref.xmax + xc3_xy[0]/2 - 13*2)
# Row 2 (Middle): Big MIM Capacitor Array
# xc3_ref.move((0, 0))

xm6_ref.movey(xm6_xy[1]/2)
xm2_ref.movey(xm2_xy[1]/2)
xm3_ref.movey(xm3_xy[1]/2)
xc3_ref.movey(xc3_xy[1]/2 - 12)

xm_max = max(xm6_xy[1], xm2_xy[1], xm3_xy[1])
separacion = 5


xm1_ref.movex(xm3_ref.xmax + 3)
xm1_ref.movey(xm_max + separacion + xm1_xy[1]/2)





In [ ]:
display_component(integrator, scale = 1, path=".")

In [ ]:
# =========================================================================
# 4. ADD AVDD & AVSS POWER RAILS AND CONNECT SUPPLY NODES
# =========================================================================

# https://gemini.google.com/app/131a537106def8cd
# Calculate full horizontal span of the cell to draw supply rails
bbox = evaluate_bbox(integrator) #calcular el tamaño del componente
top_y = integrator.ymax + 3.0
bottom_y = integrator.ymin - 3.0

# Draw horizontal metal2 power rails
avdd_rail = integrator << gf.components.rectangle(
    size=(bbox[0]+4, 1.0), 
    layer=pdk.get_layer("metal2")
)
avdd_rail.move((-2, top_y))

avss_rail = integrator << gf.components.rectangle(
    size=(bbox[0]+4, 1.0), 
    layer=pdk.get_layer("metal2")
)
avss_rail.move((-2, bottom_y))

#adding a VIA M2M3 to the avdd avss pin, moving to the center of the rail
viam2m3 = via_stack(pdk, "met2", "met3", centered=True) #met2 is the bottom layer. met3 is the top layer.
via_avdd = integrator << viam2m3
via_avdd.move((bbox[0]/2, top_y + 0.5))
via_avss = integrator << viam2m3
via_avss.move((bbox[0]/2, bottom_y + 0.5))

# Add global ports for the supply rails
integrator.add_port("avdd", center=(bbox[0]/2, top_y + 0.5), width=1.0, orientation=180, layer=pdk.get_layer("metal3"), port_type="electrical")
integrator.add_port("avss", center=(bbox[0]/2, bottom_y + 0.5), width=1.0, orientation=180, layer=pdk.get_layer("metal3"), port_type="electrical")
# add label to those global ports 
integrator.add_label(text="avdd", position=(bbox[0]/2, top_y + 0.5), layer=pdk.get_glayer("met3_label") , magnification=1.5)
integrator.add_label(text="avss", position=(bbox[0]/2, bottom_y + 0.5), layer=pdk.get_glayer("met3_label") , magnification=1.5)

In [ ]:
# =========================================================================
# 5. ADD MET2 PHYSICAL PINS AND LABELS FOR LVS
# =========================================================================

# Map port names to their source port objects
pins_to_create = {
    "Vext_pin": xm6_ref.ports["gate_E"],
    "i_50": xm3_ref.ports["multiplier_0_drain_E"],
}


for pin_name, port in pins_to_create.items():
    # Extract center position, size, and orientation from port object
    center = port.center
    width = port.width
    height = port.width  # Standard square pin area around the port center


    #adding a via to each pin port to have them into metal 3
    viam2m3 = via_stack(pdk, "met2", "met3", centered=True) #met2 is the bottom layer. met3 is the top layer.
    via_port = integrator << viam2m3
    via_port.move((center[0] - width / 2, center[1]))

    integrator.add_port(pin_name, center=(center[0] - width / 2, center[1]), width=1.0, orientation=180, layer=pdk.get_layer("metal3"),  port_type="electrical") 

    # 2. Add text label on met2_pin layer for net identity
    integrator.add_label(
        text=pin_name, position=(center[0] - width / 2, center[1]), layer=pdk.get_glayer("met3_label"), magnification=1.2
    )
    


In [ ]:
display_component(integrator, scale = 1, path=".")

In [ ]:
# 4. Connecting vdd and vss rails
# ----------------------------------------------------------------------

integrator << L_route(pdk,xm1_ref.ports["tie_E_top_met_N"], via_avdd.ports["bottom_met_W"])

integrator << L_route(pdk,xm2_ref.ports["tie_E_top_met_S"], via_avss.ports["bottom_met_W"])
integrator << L_route(pdk,xm3_ref.ports["tie_W_top_met_S"], via_avss.ports["bottom_met_W"])
integrator << L_route(pdk,xm6_ref.ports["tie_E_top_met_S"], via_avss.ports["bottom_met_W"])

integrator << c_route(pdk,xm6_ref.ports["multiplier_0_drain_W"], via_avdd.ports["bottom_met_W"])
integrator << straight_route(pdk,xc3_ref.ports["row0_col0_array_row0_col0_bottom_met_S"], via_avss.ports["bottom_met_W"])

integrator << straight_route(pdk, xm2_ref.ports["multiplier_0_source_E"], xm2_ref.ports["tie_E_top_met_N"])

integrator << straight_route(pdk, xm3_ref.ports["multiplier_0_source_E"], xm3_ref.ports["tie_E_top_met_N"])

In [ ]:
# 4. Routing Net Connections
# ----------------------------------------------------------------------


viam2m3 = via_stack(pdk, "met2", "met3", centered=True)
vg_via = integrator << viam2m3
vg_via.move(xm1_ref.ports["multiplier_0_gate_S"].center).movey(-1.5)
integrator << straight_route(pdk, vg_via.ports["top_met_N"],xm1_ref.ports["multiplier_0_gate_S"] )
integrator << c_route(pdk, xm1_ref.ports["multiplier_0_drain_W"], vg_via.ports["bottom_met_W"])
integrator << L_route(pdk, xm2_ref.ports["multiplier_0_drain_W"], vg_via.ports["top_met_S"])

viam2m31 = via_stack(pdk, "met2", "met3", centered=True)
vg_via1 = integrator << viam2m31
vg_via1.move(xm3_ref.ports["multiplier_0_gate_E"].center).movex(2)
integrator << straight_route(pdk, xm3_ref.ports["multiplier_0_gate_E"], vg_via1.ports["top_met_N"])
integrator << L_route(pdk, xm3_ref.ports["multiplier_0_drain_E"], vg_via1.ports["top_met_N"])
integrator << straight_route(pdk, xm2_ref.ports["multiplier_0_gate_W"], vg_via1.ports["top_met_E"])

viam2m32 = via_stack(pdk, "met2", "met3", centered=True)
vg_via2 = integrator << viam2m32
vg_via2.move(xm1_ref.ports["multiplier_0_source_W"].center).movex(-8)
integrator << L_route(pdk, xm6_ref.ports["multiplier_0_source_E"], vg_via2.ports["top_met_N"])
integrator << straight_route(pdk, xm1_ref.ports["multiplier_0_source_W"], vg_via2.ports["top_met_N"])
integrator << L_route(pdk, vg_via2.ports["top_met_N"], xc3_ref.ports["row1_col0_top_met_W"])



In [ ]:
display_component(integrator, scale = 1, path=".")

In [ ]:
display_component(integrator, scale = 1, path=".")

In [ ]:
integrator.name="integrator"
drc_result = gf180.drc_magic(integrator, integrator.name)

In [ ]:
import os
from pathlib import Path
import tempfile
magicrc_file = Path(os.environ['PDKPATH']) / "libs.tech" / "magic" / f"{os.environ['PDK']}.magicrc"
design_name=integrator.name
path_to_dir = "/foss/designs/libs/snn_analog/Integrator/"

pex_path = path_to_dir + f"{design_name}.spice"
gds_path = path_to_dir + f"{design_name}.gds"

integrator.write_gds(str(gds_path))
    
magic_script_content = f"""
drc off            
gds flatglob *\\$\\$*
gds read {gds_path}

flatten {design_name}
load {design_name}
select top cell
extract do local
extract all
ext2sim labels on
ext2sim
extresist tolerance 10
extresist
ext2spice lvs
ext2spice cthresh 0
ext2spice extresist on
ext2spice -o {str(pex_path)}
exit
"""

# magic_script_content = f"""
# drc off            
# gds flatglob *\\$\\$*
# gds read {gds_path}
# load {design_name}
# select top cell
# extract do local
# extract all
# ext2sim labels on
# ext2sim
# ext2spice lvs
# ext2spice -o {str(pex_path)}
# exit
# """

with tempfile.NamedTemporaryFile(mode='w', delete=False) as magic_script_file:
    magic_script_file.write(magic_script_content)
    magic_script_path = magic_script_file.name
    
magic_cmd = f"bash -c 'magic -rcfile {magicrc_file} -noconsole -dnull < {magic_script_path}'",
magic_subproc = subprocess.run(
    magic_cmd, 
    shell=True,
    check=True,
    capture_output=True
)

magic_subproc_code = magic_subproc.returncode
magic_subproc_out = magic_subproc.stdout.decode('utf-8')
print(magic_subproc_out)

In [ ]:
import glob
extensions = [
            "els"
            "*.gds",
            "*.ext",
            "*.res.ext",
            "*.lvs.rpt",
            "*_lvs.rpt",
            "*.nodes",
            "*.sim",
            "*.pex.spice",
            "*_pex.spice"
            ]
files_to_delete = []
for ext in extensions:
    files_to_delete.extend(glob.glob(ext))
    
# Delete the files
for file_path in files_to_delete:
    try:
        os.remove(file_path)
        print(f"Deleted: {file_path}")
    except OSError as e:
        print(f"Error deleting {file_path}: {e}")

In [ ]:
import os
from pathlib import Path

gf180.lvs_netgen(
    layout=integrator,
    design_name = integrator.name,
    pdk_root = Path(os.environ['PDKPATH']),
    lvs_setup_tcl_file = Path(os.environ['PDKPATH']) / "libs.tech" / "netgen" / f"{os.environ['PDK']}_setup.tcl",
    lvs_schematic_ref_file = Path(str(path_to_dir + "integrator.spice")),
    netlist = Path(str(path_to_dir + "integrator_sch.spice")),
    output_file_path =  Path(str(path_to_dir))
)